<h2>File paths and imports</h2>

This steps is important because it tells the program where it can access the files needed throughout the process.

You must specify 3 paths:

<li> <b>model_path</b>: body detection model
<li> <b>input_video_directory</b>: Folder containing the input videos
<li> <b>output_video_directory</b>: Folder where the treated videos will be redirected to

You can also mention videos that you don't want to be treated. To do so, simply indicate their name(s) without the extension in <b>ignore_S1</b> and <b>ignore_S2</b>.

In [ ]:
import os
import sys
from ui_lib import *

# ==============================================================================
# CONFIGURATION: TUNE YOUR MODEL HERE
# ==============================================================================
# CHANGE THIS TO "v2_tuned" WHEN YOU RUN THIS NEW CODE
MODEL_VERSION = "v2_tuned" 

# 1. TUNING DEEPSORT (Fixes AssA / ID Switching)
DEEPSORT_MAX_AGE = 70    # Default is usually 30. Increased to 70 frames (~2.5s) to handle occlusions.
DEEPSORT_N_INIT = 3      # Default is 3. Number of hits before confirming a track.
DEEPSORT_MAX_DIST = 0.5  # Cosine distance threshold.

# 2. FILTERING FALSE POSITIVES (Fixes DetPr)
YOLO_CONFIDENCE = 0.7    # Increased from 0.5 to 0.7 to remove "ghost" detections.


# Paths
model_path = "../../../Models/Body Detection/Body_detection_model.pt"

input_video_directory = "input"
output_video_directory = "output"
temp_directory = f"{output_video_directory}/temp"
raw_text_output_directory = f"{temp_directory}/raw_output"
manual_annotations_directory = f"{input_video_directory}/manual_annotations"
treated_directory = f"{output_video_directory}/treated"
final_directory = f"{output_video_directory}/final"

# Videos to ignore per step
ignore_S1 = [
    "example_1",
    "example_2",
    "20241009 - 09h07.MP4",
    "20241009 - 09h07",
    "20241015 - 12h41-(tempCut)",
    "big.MP4",
    "big",
    "20241019 - 13h28-(temp)",
    "20241019 - 14h29-(temp)",
    "20241019 - 13h28",
]

ignore_S2 = [
    "example_1",
    "example_2",
    "20241009 - 09h07.MP4",
    "20241009 - 09h07",
    "big.MP4",
    "big",
    "20241015 - 12h41-(tempCut)",
    "20241019 - 13h28-(temp)",
]

ignore_S3 = [
    "example_1",
    "example_2",
    "20241009 - 09h07.MP4",
    "20241009 - 09h07",
    "big.MP4",
    "big",
]

# Helper: create required folders
for path in [
    input_video_directory,
    output_video_directory,
    temp_directory,
    raw_text_output_directory,
    manual_annotations_directory,
    treated_directory,
    final_directory,
]:
    os.makedirs(path, exist_ok=True)

# DeepSORT/YOLO setup
nn_budget = None
metric = nn_matching.NearestNeighborDistanceMetric("cosine", DEEPSORT_MAX_DIST, nn_budget)

DeepSort = DeepSortTracker(
    metric, 
    max_iou_distance=0.7, 
    max_age=DEEPSORT_MAX_AGE, 
    n_init=DEEPSORT_N_INIT
)

YOLOv8s = YOLO(model_path)
Osnet = torchreid.models.build_model(name="osnet_x1_0", num_classes=751, pretrained=True)
Osnet.eval()

def has_audio_stream(video_path: str) -> bool:
    """
    Optional: quick check to avoid mux when there is no audio.
    If your mux_audio already tolerates missing audio, you can skip this.
    """
    try:
        import subprocess, json
        probe_cmd = [
            "ffprobe",
            "-v", "error",
            "-select_streams", "a",
            "-show_entries", "stream=index",
            "-of", "json",
            video_path,
        ]
        out = subprocess.check_output(probe_cmd).decode("utf-8")
        data = json.loads(out)
        streams = data.get("streams", [])
        return len(streams) > 0
    except Exception:
        # Fallback: assume audio exists to keep behavior; mux_audio should handle errors gracefully
        return True

/home/diego/.pyenv/versions/chimprec310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/diego/.pyenv/versions/chimprec310/lib/python3.10/site-packages/torchreid/reid/metrics/rank.py:11: UserWarning: Cython evaluation (very fast so highly recommended) is unavailable, now use python evaluation.
  warnings.warn(


Successfully loaded imagenet pretrained weights from "/home/diego/.cache/torch/checkpoints/osnet_x1_0_imagenet.pth"
** The following layers are discarded due to unmatched keys or layer size: ['classifier.weight', 'classifier.bias']


<h2>First step:</h2>

This step will process the input videos automatically. In other words, it will draw rectangles around each individuals and track them throughout the video.

In [2]:
for input_video in os.listdir(input_video_directory):
    if not (input_video.endswith(".mp4") or input_video.endswith(".MP4")):
        continue

    full_video_path = os.path.join(input_video_directory, input_video)
    video_name = os.path.splitext(input_video)[0]

    if video_name in ignore_S1:
        print(f"{video_name}.mp4 ignored")
        continue

    annotation_file_path = f"{manual_annotations_directory}/{video_name}.txt"
    try:
        with open(annotation_file_path, "x") as f:
            print(f"{video_name}.txt automatically created in {manual_annotations_directory}.")
    except FileExistsError:
        print(f"{video_name}.txt already present in {manual_annotations_directory}.")
    print()

    raw_txt_path = f"{raw_text_output_directory}/{video_name}.txt"

    # Tracking
    print(f"--- Processing {video_name} with {MODEL_VERSION} parameters ---")
    print(f"Conf: {YOLO_CONFIDENCE}, Max_Age: {DEEPSORT_MAX_AGE}")
    perform_tracking(
        input_video_path=full_video_path,
        output_text_file_path=raw_txt_path,
        detection_model=YOLOv8s,
        tracker=DeepSort,
        confidence_threshold=YOLO_CONFIDENCE,
        model_feature_extraction=Osnet,
    )
    print(f"Annotations ready for video: {full_video_path}.\n")

    processed_with_audio = f"{temp_directory}/{video_name}-(temp)-audio.mp4"

    # Draw and mux (only one output, with audio when available)
    draw_bbox_from_file(
        file_path=raw_txt_path,
        input_video_path=full_video_path,
        output_video_path=processed_with_audio,
        annotation_type="bbox",
        draw_frame_count=True,
    )

    if has_audio_stream(full_video_path):
        print("Adding audio...")
        mux_audio(full_video_path, processed_with_audio, processed_with_audio)
    else:
        print("No audio stream detected; skipping mux.")
    print(f"Treatment done: {full_video_path}.\n")

20241009 - 09h07.mp4 ignored
20241015 - 12h41-(tempCut).mp4 ignored
big.mp4 ignored
20241019 - 14h29-(temp).mp4 ignored
20241019 - 13h28.txt automatically created in input/manual_annotations.

--- Processing 20241019 - 13h28 with v2_tuned parameters ---
Conf: 0.7, Max_Age: 70


Tracking progress (20241019 - 13h28.MP4): 100%|██████████| 55128/55128 [1:13:26<00:00, 12.51it/s]  


Annotations ready for video: input/20241019 - 13h28.MP4.



Drawing annotations (20241019 - 13h28.MP4): 100%|██████████| 55128/55128 [18:55<00:00, 48.53it/s]


Adding audio...


ffmpeg version n8.0.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with gcc 15.2.1 (GCC) 20260103
  configuration: --prefix=/usr --disable-debug --disable-static --disable-stripping --enable-amf --enable-avisynth --enable-cuda-llvm --enable-lto --enable-fontconfig --enable-frei0r --enable-gmp --enable-gnutls --enable-gpl --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libdav1d --enable-libdrm --enable-libdvdnav --enable-libdvdread --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgsm --enable-libharfbuzz --enable-libiec61883 --enable-libjack --enable-libjxl --enable-libmodplug --enable-libmp3lame --enable-libopencore_amrnb --enable-libopencore_amrwb --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libplacebo --enable-libpulse --enable-librav1e --enable-librsvg --enable-librubberband --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libsvtav1 --enab

Treatment done: input/20241019 - 13h28.MP4.



[out#0/mp4 @ 0x557a17430e00] video:2967999KiB audio:206730KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.023783%
frame=55128 fps=15485 q=-1.0 Lsize= 3175484KiB time=00:18:22.56 bitrate=23593.8kbits/s speed= 310x elapsed=0:00:03.56    


<h2>Second step:</h2>

This final step will take into account your modifications to modify the output of the automated process.

<b>If you need to modify annotations previously created:</b> simply run this part of the code. In this case, there's no need to run the above cells.

In [2]:
# ---------- STEP 2: apply manual edits -> treated output (per-video folder) ----------
for input_video in os.listdir(input_video_directory):
    if not (input_video.endswith(".mp4") or input_video.endswith(".MP4")):
        continue

    full_video_path = os.path.join(input_video_directory, input_video)
    video_name = os.path.splitext(input_video)[0]

    if video_name in ignore_S2:
        print(f"{video_name}.mp4 ignored")
        continue

    annotation_file = f"{manual_annotations_directory}/{video_name}.txt"
    raw_reader = raw_tracking_data_reader(f"{raw_text_output_directory}/{video_name}.txt")

    try:
        edit_reader = modification_reader(annotation_file)
    except Exception:
        print(
            f"Error: the manual annotation file related to the video <{full_video_path}> is not found. "
            f"It must be located at <{annotation_file}>."
        )
        continue

    # per-video output folder
    video_out_dir = os.path.join(treated_directory, video_name)
    os.makedirs(video_out_dir, exist_ok=True)

    metadata_file_path = os.path.join(video_out_dir, f"{video_name}-treated.txt")
    output_video_path = os.path.join(video_out_dir, f"{video_name}-treated.mp4")
    writer = data_writer(metadata_file_path)

    modified_data = edit_raw_output(raw_reader, edit_reader)
    writer.write(modified_data)

    draw_bbox_from_file(
        file_path=metadata_file_path,
        input_video_path=full_video_path,
        output_video_path=output_video_path,
        annotation_type="bbox",
        draw_frame_count=True,
    )

    if has_audio_stream(full_video_path):
        print("Adding audio...")
        mux_audio(full_video_path, output_video_path, output_video_path)
    else:
        print("No audio stream detected; skipping mux.")
    print(f"Treatment done: {full_video_path}.\n")

20241009 - 09h07.mp4 ignored
20241015 - 12h41-(tempCut).mp4 ignored
big.mp4 ignored
20241019 - 13h28-(temp).mp4 ignored


Drawing annotations (20241019 - 14h29-(temp).mp4): 100%|██████████| 55032/55032 [15:26<00:00, 59.42it/s]


No audio stream detected; skipping mux.
Treatment done: input/20241019 - 14h29-(temp).mp4.



<h2> Third Step: </h2>

This final step will only be used to generate the arrows associated with the corresponding chimpanzees.<br>
<b>It's only to be performed when the manual annotations are 100% correct.</b>


In [4]:
# ---------- STEP 3: final arrows/names -> final output (per-video folder) ----------
for input_video in os.listdir(input_video_directory):
    if not (input_video.endswith(".mp4") or input_video.endswith(".MP4")):
        continue

    full_video_path = os.path.join(input_video_directory, input_video)
    video_name = os.path.splitext(input_video)[0]

    if video_name in ignore_S3:
        print(f"{video_name}.mp4 ignored")
        continue

    annotation_file = f"{manual_annotations_directory}/{video_name}.txt"
    raw_reader = raw_tracking_data_reader(f"{raw_text_output_directory}/{video_name}.txt")

    try:
        edit_reader = modification_reader(annotation_file)
    except Exception:
        print(
            f"Error: the manual annotation file related to the video <{full_video_path}> is not found. "
            f"It must be located at <{annotation_file}>."
        )
        continue

    # per-video output folder
    video_out_dir = os.path.join(final_directory, video_name)
    os.makedirs(video_out_dir, exist_ok=True)

    metadata_file_path = os.path.join(video_out_dir, f"{video_name}-final.txt")
    output_video_path = os.path.join(video_out_dir, f"{video_name}-final.mp4")
    writer = data_writer(metadata_file_path)

    modified_data = edit_raw_output(raw_reader, edit_reader)
    writer.write(modified_data)

    draw_bbox_from_file(
        file_path=metadata_file_path,
        input_video_path=full_video_path,
        output_video_path=output_video_path,
        annotation_type="triangle",
        draw_frame_count=False,
    )

    if has_audio_stream(full_video_path):
        print("Adding audio...")
        mux_audio(full_video_path, output_video_path, output_video_path)
    else:
        print("No audio stream detected; skipping mux.")
    print(f"Treatment done: {full_video_path}.\n")

20241009 - 09h07.mp4 ignored


Drawing annotations (20241015 - 12h41-(tempCut).mp4): 100%|██████████| 484/484 [00:08<00:00, 58.92it/s]
ffmpeg version n8.0.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with gcc 15.2.1 (GCC) 20251112
  configuration: --prefix=/usr --disable-debug --disable-static --disable-stripping --enable-amf --enable-avisynth --enable-cuda-llvm --enable-lto --enable-fontconfig --enable-frei0r --enable-gmp --enable-gnutls --enable-gpl --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libdav1d --enable-libdrm --enable-libdvdnav --enable-libdvdread --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgsm --enable-libharfbuzz --enable-libiec61883 --enable-libjack --enable-libjxl --enable-libmodplug --enable-libmp3lame --enable-libopencore_amrnb --enable-libopencore_amrwb --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libplacebo --enable-libpulse --enable-librav1e --enable-librsvg --enable-librubberband --enabl

Adding audio...
Treatment done: input/20241015 - 12h41-(tempCut).mp4.



Drawing annotations (big.MP4):   0%|          | 27/16392 [00:00<04:16, 63.86it/s]
ffmpeg version n8.0.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with gcc 15.2.1 (GCC) 20251112
  configuration: --prefix=/usr --disable-debug --disable-static --disable-stripping --enable-amf --enable-avisynth --enable-cuda-llvm --enable-lto --enable-fontconfig --enable-frei0r --enable-gmp --enable-gnutls --enable-gpl --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libdav1d --enable-libdrm --enable-libdvdnav --enable-libdvdread --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgsm --enable-libharfbuzz --enable-libiec61883 --enable-libjack --enable-libjxl --enable-libmodplug --enable-libmp3lame --enable-libopencore_amrnb --enable-libopencore_amrwb --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libplacebo --enable-libpulse --enable-librav1e --enable-librsvg --enable-librubberband --enable-libsnappy --enable-l

Adding audio...
Treatment done: input/big.MP4.



[out#0/mp4 @ 0x55d914bce800] video:1218KiB audio:61470KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.003895%
frame=   27 fps=0.0 q=-1.0 Lsize=   62691KiB time=00:00:00.54 bitrate=951038.8kbits/s speed= 4.3x elapsed=0:00:00.12    
